# Prompt 3: Lazy Evaluation & Transformation vs Action
**Databricks Certified Associate Developer for Apache Spark — Topic 1 (20%)**

---

## Overview

Lazy evaluation is one of Spark's core design decisions. Understanding it explains *why* Spark can optimise queries before running them and *how* the execution model works.

| Concept | Key idea |
|---|---|
| **Transformation** | Describes *what* to do — builds a logical plan, nothing runs |
| **Action** | *Triggers* execution — Spark materialises the result |
| **Lazy evaluation** | Transformations are deferred until an Action is called |
| **DAG** | The accumulated transformations form a Directed Acyclic Graph of stages |

---
## 1. What are Transformations?

A **transformation** is any operation that returns a new DataFrame (or RDD) without immediately computing a result.
Spark records the transformation in a **logical plan** — a description of what to do — but does **no actual work**.

### Narrow vs Wide Transformations

| Type | Definition | Shuffle? | Examples |
|---|---|---|---|
| **Narrow** | Each output partition depends on exactly one input partition | No | `filter`, `select`, `withColumn`, `map`, `flatMap`, `union` |
| **Wide** | Output partition depends on multiple input partitions | **Yes** | `groupBy`, `join`, `distinct`, `orderBy`, `repartition` |

> **Exam tip**: Wide transformations create **stage boundaries** in the DAG. Narrow transformations are pipelined within a single stage.

### Common Transformation Examples

```python
df2 = df.filter(df.age > 30)          # narrow  — no shuffle
df3 = df2.select('name', 'salary')    # narrow  — no shuffle
df4 = df3.withColumn('tax', df3.salary * 0.2)  # narrow
df5 = df4.groupBy('dept').agg(...)    # wide    — shuffle boundary
df6 = df.join(other_df, 'id')         # wide    — shuffle boundary
df7 = df.orderBy('salary')            # wide    — shuffle boundary
```

None of the lines above execute any computation — they only build the plan.

---
## 2. What are Actions?

An **action** is an operation that **triggers execution** of the accumulated logical plan.
When Spark sees an action it:
1. Optimises the logical plan (Catalyst optimizer)
2. Builds a physical plan
3. Schedules and runs Tasks on Executors
4. Returns the result

### Common Action Examples

| Action | What it returns | Notes |
|---|---|---|
| `collect()` | List of all rows to the Driver | Dangerous on large data — OOM risk |
| `count()` | Integer row count | Triggers full scan |
| `show(n)` | Prints first n rows (default 20) | Triggers partial execution |
| `first()` / `head()` | First row as a Row object | Stops after first partition (usually) |
| `take(n)` | List of first n rows | Similar to head |
| `write` | Saves data to storage | Full execution — common in production |
| `foreach()` / `foreachPartition()` | Applies function to each row/partition | Side-effect action |
| `toPandas()` | Pandas DataFrame | Pulls all data to Driver — use carefully |
| `describe()` | Summary statistics DataFrame | Triggers execution |

> **Exam tip**: `printSchema()` is **NOT an action** — it only inspects the schema metadata, which Spark knows without executing the plan.

---
## 3. Why Spark Uses Lazy Evaluation

### Benefits

1. **Query optimisation**: By deferring execution, the Catalyst optimizer sees the *entire* plan and can reorder, combine, or eliminate operations before running anything.
   - Example: a `filter` written after a `join` can be pushed *before* the join (predicate pushdown), drastically reducing data shuffled.

2. **Pipelining narrow transformations**: Multiple narrow transforms on the same partition are collapsed into a single pass (one Task reads the data once and applies all transforms).

3. **Fault tolerance**: Because the full lineage (chain of transformations) is recorded, Spark can recompute lost partitions by replaying only the necessary transformations from the source.

4. **Avoid unnecessary computation**: If only part of the result is needed (`take(5)`, `first()`), Spark may execute only enough Tasks to satisfy the action.

```
Without lazy evaluation (eager):
  filter()  →  run  →  select()  →  run  →  groupBy()  →  run
  3 separate data passes, no cross-operation optimisation

With lazy evaluation:
  filter() + select() + groupBy()  →  optimise → run once
  Catalyst may push filter before groupBy, pipeline filter+select
```

In [ ]:
from pyspark.sql import SparkSession
import time

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("LazyEvaluationDemo")
    .getOrCreate()
)
print("SparkSession ready")

In [ ]:
# ── Demonstrating lazy evaluation ─────────────────────────────────────────
# Create sample data
data = [
    (1, "Alice",   "Engineering", 95000),
    (2, "Bob",     "Engineering", 80000),
    (3, "Carol",   "Marketing",   70000),
    (4, "David",   "Marketing",   65000),
    (5, "Eve",     "Engineering", 110000),
    (6, "Frank",   "HR",          60000),
    (7, "Grace",   "HR",          55000),
]
df = spark.createDataFrame(data, ["id", "name", "dept", "salary"])

# Stack several transformations — NOTHING RUNS YET
print("Building transformation chain...")
t0 = time.time()

step1 = df.filter(df.salary > 60000)                   # narrow — no execution
step2 = step1.select("name", "dept", "salary")         # narrow — no execution
step3 = step2.withColumn("bonus", step2.salary * 0.1)  # narrow — no execution
step4 = step3.orderBy("salary", ascending=False)       # wide   — no execution

t1 = time.time()
print(f"Time to build plan (4 transforms): {(t1-t0)*1000:.1f} ms — nothing ran!")

# NOW trigger with an action
print("\nCalling show() — this triggers execution:")
t2 = time.time()
step4.show()   # <── ACTION: plan is optimised and executed here
t3 = time.time()
print(f"Time to execute (all 4 transforms): {(t3-t2)*1000:.1f} ms")

In [ ]:
# ── Each action triggers a NEW execution of the same plan ─────────────────
# (unless the DataFrame is cached — see below)

print("First action — count():")
print(f"  Row count: {step4.count()}")   # executes the plan

print("\nSecond action — collect():")
rows = step4.collect()                    # executes the plan AGAIN from scratch
print(f"  Rows collected: {len(rows)}")

print("\nEach action re-executes the full lineage unless the DataFrame is cached.")
print("Use df.cache() or df.persist() to avoid recomputing expensive plans.")

In [ ]:
# ── printSchema() is NOT an action ────────────────────────────────────────
# It inspects metadata only — does not trigger execution

print("printSchema() — no execution, just schema metadata:")
step4.printSchema()

print("\nexplain() — shows the optimised plan (no execution):")
step4.explain()   # shows physical plan — does not run it

In [ ]:
# ── Catalyst optimisation in action: predicate pushdown ───────────────────
# Spark rewrites the plan to apply filters as early as possible

from pyspark.sql.functions import col

# Filter AFTER join (as written by developer)
dept_data = [("Engineering", "NYC"), ("Marketing", "LA"), ("HR", "Chicago")]
dept_df = spark.createDataFrame(dept_data, ["dept", "location"])

joined = df.join(dept_df, on="dept", how="inner")
filtered_after = joined.filter(col("salary") > 90000)

print("Plan WITH filter written after join (Catalyst pushes it before):")
filtered_after.explain()
# Note: in the physical plan Spark will apply the salary filter BEFORE the join
# This reduces the number of rows shuffled during the join

In [ ]:
# ── Narrow vs Wide transformations visualised ──────────────────────────────
from pyspark.sql.functions import sum as spark_sum, avg, count

# Stage 1: Narrow transforms pipelined into one stage
# Stage 2: Wide transform (groupBy) creates a new stage after shuffle

result = (
    df
    .filter(df.salary > 60000)                         # narrow — Stage 1
    .withColumn("bonus", df.salary * 0.1)              # narrow — Stage 1
    .groupBy("dept")                                    # wide   — Stage boundary
    .agg(
        count("*").alias("headcount"),
        avg("salary").alias("avg_salary"),
        spark_sum("bonus").alias("total_bonus")
    )
    .orderBy("avg_salary", ascending=False)             # wide   — Stage boundary
)

print("Execution plan (2 wide transforms = 3 stages):")
result.explain()
print()
result.show()

In [ ]:
# ── Caching to avoid re-execution ─────────────────────────────────────────
# Cache an intermediate DataFrame that is used by multiple actions

expensive_df = (
    df
    .filter(df.salary > 60000)
    .withColumn("bonus", df.salary * 0.1)
    .join(dept_df, on="dept")
)

expensive_df.cache()   # marks for caching — computed on first action

print("First action — triggers execution AND caches result:")
print(f"  count = {expensive_df.count()}")

print("\nSecond action — uses cached result (no recomputation):")
expensive_df.show()

expensive_df.unpersist()   # release cache when done
print("\nCache released")

---
## 4. Transformation vs Action — Complete Reference

### Transformations (lazy — return a new DataFrame)

| Operation | Narrow/Wide | Notes |
|---|---|---|
| `filter()` / `where()` | Narrow | Row predicate |
| `select()` / `selectExpr()` | Narrow | Column projection |
| `withColumn()` | Narrow | Add/replace column |
| `withColumnRenamed()` | Narrow | Rename column |
| `drop()` | Narrow | Remove columns |
| `map()` / `flatMap()` (RDD) | Narrow | Row transformation |
| `union()` / `unionAll()` | Narrow | Stack DataFrames |
| `limit()` | Narrow | Take first N rows |
| `sample()` | Narrow | Random sample |
| `groupBy().agg()` | Wide | Aggregation with shuffle |
| `join()` | Wide | Typically shuffles |
| `distinct()` | Wide | Deduplication |
| `orderBy()` / `sort()` | Wide | Global sort requires shuffle |
| `repartition(n)` | Wide | Shuffle to n partitions |
| `coalesce(n)` | Narrow* | Reduce partitions without full shuffle |
| `crossJoin()` | Wide | Cartesian product |

### Actions (eager — trigger execution)

| Operation | Returns | Risk |
|---|---|---|
| `collect()` | List[Row] | OOM if data is large |
| `count()` | int | Full scan |
| `show(n)` | None (prints) | Safe for exploration |
| `first()` / `head()` | Row | Returns 1 row |
| `take(n)` | List[Row] | Returns n rows |
| `toPandas()` | pandas DataFrame | OOM on large data |
| `write.*(path)` | None | Full execution |
| `foreach()` | None | Side-effect per row |
| `reduce()` (RDD) | single value | Full scan |

---
## 5. Real-World Scenarios

<details>
<summary>Scenario 1: Filter pushed before join — Catalyst optimisation saves 10× shuffle cost</summary>

**Situation**: An analyst writes a query that joins a 10 billion-row transactions table with a 1 million-row customers table, then filters for customers in one city. The first run takes 45 minutes.

**Root cause**: The developer wrote the filter after the join. Without understanding lazy evaluation they assume the join runs first, then the filter.

**What Catalyst does**: Because of lazy evaluation, Spark sees the full plan before executing. It recognises the filter only touches the customers table and rewrites the plan to filter customers *before* the join. The join now sees 5,000 rows instead of 1 million.

```python
transactions = spark.read.parquet("/data/transactions")    # 10 billion rows
customers    = spark.read.parquet("/data/customers")       # 1 million rows

# Developer writes filter after join — looks expensive
result = (
    transactions
    .join(customers, on="customer_id")
    .filter(customers.city == "London")    # Catalyst pushes this BEFORE the join
)

# Verify the optimised plan
result.explain()   # physical plan shows filter applied before join
result.write.parquet("/output/london_txns")   # action — executes optimised plan
```

**Expected outcome**: Job completes in ~4 minutes instead of 45 because far fewer rows are shuffled during the join.

**Exam sub-topic**: Lazy evaluation enables Catalyst predicate pushdown
</details>

<details>
<summary>Scenario 2: Accidental double execution due to no caching</summary>

**Situation**: A pipeline reads and processes a large DataFrame, then calls `count()` to log the row count, then calls `write()` to save it. Runtime is 2× longer than expected.

**Root cause**: Without caching, each action (`count()` and `write()`) re-executes the entire transformation plan from scratch — reading the source data twice and reapplying all transforms twice.

```python
processed = (
    spark.read.parquet("/raw/events")    # expensive read
    .filter(...)                          # transforms
    .withColumn(...)
    .join(lookup_df, "id")
)

# BAD: two actions on the same uncached DataFrame = plan executed twice
row_count = processed.count()            # execution #1
processed.write.parquet("/output/")     # execution #2 — reads source again!

# FIX: cache before the first action
processed.cache()
row_count = processed.count()            # execution #1 — result cached
processed.write.parquet("/output/")     # uses cache — no re-read
processed.unpersist()
```

**Expected outcome**: After caching, runtime drops by ~50% because source is read once.

**Exam sub-topic**: Actions trigger full re-execution; caching avoids recomputation
</details>

<details>
<summary>Scenario 3: printSchema() mistakenly used to check data quality</summary>

**Situation**: A new Spark developer calls `df.printSchema()` and `df.explain()` expecting that any data quality errors will be surfaced. No errors appear so they assume the pipeline is correct. In production, `write()` fails with a schema mismatch error.

**Root cause**: `printSchema()` and `explain()` are not actions. They inspect the *logical schema* without reading any data. Type errors in actual data (e.g., a string in an integer column) are only surfaced when an action reads the data.

```python
df = spark.read.schema(expected_schema).csv("/data/input.csv")

df.printSchema()    # NOT an action — only shows declared schema
df.explain()        # NOT an action — shows plan, not data

# To actually validate data, use an action:
print(f"Row count: {df.count()}")   # action — reads data, surfaces parse errors
df.show(5)                           # action — reveals actual values

# For data quality checks:
from pyspark.sql.functions import col, count, when, isnan
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df.columns
])
null_counts.show()   # action — actually reads and checks data
```

**Expected outcome**: Using actions surfaces real data issues before the pipeline reaches production.

**Exam sub-topic**: Distinguishing actions from non-executing operations; printSchema is not an action
</details>

<details>
<summary>Scenario 4: Narrow transforms pipelined — single-pass efficiency</summary>

**Situation**: A developer wants to understand why chaining 5 narrow transformations runs as fast as 1 transformation. They expect 5 data passes.

**Explanation**: Because all 5 transforms are narrow (each output partition depends on exactly one input partition), Spark pipelines them into a single stage. Each Task reads its partition once and applies all 5 transforms in a single pass through memory.

```python
from pyspark.sql.functions import upper, trim, col

result = (
    df
    .filter(col("salary") > 50000)                     # narrow
    .select("name", "dept", "salary")                  # narrow
    .withColumn("name_upper", upper(col("name")))      # narrow
    .withColumn("dept_clean", trim(col("dept")))       # narrow
    .withColumn("net_pay", col("salary") * 0.75)       # narrow
)

result.explain()   # physical plan shows ONE scan, not five
# Output: Project [salary > 50000 → upper → trim → multiply all in one pass]

result.show()      # action — single data pass applies all 5 transforms
```

**Expected outcome**: The `explain()` physical plan shows a single `Project` node covering all transforms. One data pass. No shuffles.

**Exam sub-topic**: Narrow transformation pipelining; why lazy evaluation is efficient
</details>

<details>
<summary>Scenario 5: Early termination with take(5) — Spark avoids full scan</summary>

**Situation**: A developer calls `df.orderBy('date').take(5)` expecting it to sort the entire multi-terabyte dataset first and then return 5 rows. They are surprised it returns in seconds.

**Explanation**: `take(n)` is an action. Spark is *lazy* — it knows only 5 rows are needed. For operations like `filter` + `take(5)`, Spark will scan only enough partitions to satisfy the request and short-circuit. For `orderBy` + `take(5)`, Spark uses a partial sort optimisation: each Task finds its local top-5, then the Driver merges them — far cheaper than a full global sort.

```python
# Scenario A: filter + take — partial scan
df_large = spark.read.parquet("/data/billions_of_rows.parquet")

five_rows = (
    df_large
    .filter(df_large.status == "ERROR")   # transformation
    .take(5)                               # action — short-circuits after 5 matches
)
print(five_rows)   # very fast — doesn't scan all partitions if 5 errors found early

# Scenario B: orderBy + take — TakeOrderedAndProject optimisation
top5_salaries = (
    df
    .orderBy("salary", ascending=False)
    .take(5)                               # TakeOrderedAndProject in physical plan
)
# Each executor finds its local top-5, driver merges — no full sort needed

df_large.createOrReplaceTempView("big_table")
spark.sql("SELECT * FROM big_table WHERE status='ERROR' LIMIT 5").show()
# LIMIT in SQL triggers the same short-circuit optimisation
```

**Expected outcome**: `take(5)` returns quickly because Spark's lazy model allows it to plan for the minimum necessary computation.

**Exam sub-topic**: Actions and short-circuit optimisation; lazy evaluation avoids unnecessary work
</details>

---
## 6. Quick Reference Summary

```
Transformation chain
─────────────────────────────────────────────────────────
read()  →  filter()  →  select()  →  groupBy()  →  action()
  │           │            │             │              │
  └──────────────────────────────────────┘              │
      Logical plan built (nothing runs)         Catalyst optimises
                                                 then DAG executes
```

### Exam Cheat Sheet

- Transformations are **lazy** — they return a new DataFrame and build the plan
- Actions are **eager** — they trigger execution and return a result or write data
- `printSchema()` and `explain()` are **not actions** — no data is read
- Narrow transforms are **pipelined** (one stage); wide transforms create **stage boundaries**
- Without caching, each action **re-executes** the full lineage from source
- Catalyst can **reorder** transforms (e.g., push filters before joins) because execution is deferred

---
## 7. Exam Practice Questions

**Q1**: Which of the following triggers execution in Spark?
```
A) df.filter(df.age > 30)
B) df.select('name')
C) df.count()
D) df.withColumn('tax', df.salary * 0.2)
```
<details><summary>Answer</summary>
**C) `df.count()`** — it is an action. All others are transformations (lazy).
</details>

---

**Q2**: A developer has a pipeline with 5 narrow transformations followed by a `groupBy().agg()`. How many stages will Spark create?

<details><summary>Answer</summary>
**2 stages**. The 5 narrow transforms are pipelined into Stage 1. The `groupBy().agg()` is a wide transform that creates a shuffle boundary, starting Stage 2.
</details>

---

**Q3**: A DataFrame is used in two separate actions without caching. How many times is the source data read?

<details><summary>Answer</summary>
**Twice** — each action triggers a full re-execution of the lineage from source. Use `df.cache()` before the first action to avoid this.
</details>

---

**Q4**: Is `df.printSchema()` an action? Does it read data from the source?

<details><summary>Answer</summary>
**No** — `printSchema()` is not an action. It only inspects the schema metadata from the logical plan. No data is read from the source.
</details>

---

**Q5**: What is the key benefit of lazy evaluation for query optimisation?

<details><summary>Answer</summary>
Lazy evaluation allows the Catalyst optimizer to see the **entire transformation plan** before executing it. This enables optimisations like predicate pushdown (applying filters before joins), column pruning (reading only needed columns), and pipelining narrow transforms — all of which are impossible with eager (immediate) execution.
</details>